# Audit sumber regional — 2026-10-05

Audit metadata seluruh JSON; tidak menilai jawaban, tidak membaca GT untuk keputusan, tidak mengubah hasil produksi. Setiap eksekusi membuat direktori baru. Pembacaan semantik dan keputusan agen dicatat terpisah dalam dokumen audit.

In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from datetime import datetime
import json, re, hashlib, unicodedata, itertools
import pandas as pd

ROOT = Path('D:/TugasAkhir')
OUT = ROOT/'result/analyses/linguistic_markers/runs'/('regional_source_audit_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
OUT.mkdir(parents=True, exist_ok=False)
print('OUTPUT', OUT, flush=True)
MODELS = sorted(p.name for p in (ROOT/'result/benchmarks/gsm8k/regional_cultural_persona').iterdir() if p.is_dir())
IDENTITIES = ['north_american','east_asian','south_asian','sub_saharan_african']
CONDITIONS = ['explicit','implicit_1','implicit_2','implicit_3']
PERSONAS = ['neutral']+[a+'_'+b for a in IDENTITIES for b in CONDITIONS]
def norm(t):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',t or '')).strip()
def sha(t):
    return hashlib.sha256(t.encode('utf-8')).hexdigest()
def save(df,name):
    df.to_csv(OUT/name,index=False,encoding='utf-8-sig')
contrasts=[]
for a in IDENTITIES:
    contrasts += [(a+'_'+c,'neutral','vs_neutral') for c in CONDITIONS]
    contrasts += [(a+'_'+c,a+'_explicit','vs_explicit') for c in CONDITIONS[1:]]
for a,b in itertools.combinations(IDENTITIES,2):
    contrasts += [(a+'_'+c,b+'_'+c,'direct') for c in CONDITIONS]
assert len(contrasts)==52
for benchmark in ['GSM8K','ETHICS']:
    dataset=json.loads((ROOT/('data/gsm8k/gsm8k_test.json' if benchmark=='GSM8K' else 'data/ethics/ETHICS_test_4subsets_10percent_from200.json')).read_text(encoding='utf-8'))
    by_id={}
    by_hash=defaultdict(list)
    for i,item in enumerate(dataset):
        subset='gsm8k' if benchmark=='GSM8K' else item['subset']
        key=f'gsm8k_test_{i:05d}' if benchmark=='GSM8K' else item['id']
        h=sha(norm(item['question'] if benchmark=='GSM8K' else item['scenario']))
        by_id[(subset,key)]=h
        by_hash[(subset,h)].append(key)
    tasks=[]
    for model in MODELS:
        for persona in PERSONAS:
            if benchmark=='GSM8K':
                base=ROOT/'result/benchmarks/gsm8k'/('Gender_Persona' if persona=='neutral' else 'regional_cultural_persona')/model/persona
                paths=sorted(base.glob('gsm8k_*.json'))
            else:
                base=ROOT/'result/benchmarks/ethics/runs/v9_main'/('Neutral Persona' if persona=='neutral' else 'Regional Cultural Persona')/model
                base=base/persona if (base/persona).is_dir() else base
                paths=sorted(base.glob('ethics_*.json'))
            assert paths,(benchmark,model,persona)
            tasks.extend((model,persona,p) for p in paths)
    def inspect(task):
        model,persona,path=task
        binary=path.read_bytes()
        error=''
        try: obj=json.loads(binary)
        except (ValueError,UnicodeError) as exc:
            obj={};error=type(exc).__name__
        subset='gsm8k' if benchmark=='GSM8K' else obj.get('subset',path.stem.split('_')[1])
        question=(obj.get('input') or {}).get('question','') if benchmark=='GSM8K' else obj.get('scenario','')
        h=sha(norm(question)) if question else ''
        ids=by_hash.get((subset,h),[])
        if h and by_id.get((subset,path.stem))==h:
            canonical,status=path.stem,'id_and_content_verified'
        elif len(ids)==1: canonical,status=ids[0],'unique_content_match'
        else: canonical,status='',('ambiguous_content' if ids else 'unknown_question')
        response=obj.get('response') or {}
        api=response.get('raw_api_response') or {}
        choices=api.get('choices',[]) if isinstance(api,dict) else []
        message=choices[0].get('message',{}) if choices else {}
        message=message if isinstance(message,dict) else {}
        final=message.get('content') if 'content' in message else response.get('content')
        final=final if isinstance(final,str) else ''
        reasoning=message.get('reasoning') or ''
        reasoning=reasoning if isinstance(reasoning,str) else ''
        success=response.get('success') is not False
        messages=(obj.get('request') or {}).get('messages') or []
        users=[m.get('content','') for m in messages if m.get('role')=='user' and isinstance(m.get('content'),str)]
        assistants=[m.get('content','') for m in messages if m.get('role')=='assistant' and isinstance(m.get('content'),str)]
        return dict(model_id=model,persona=persona,subset=subset,original_question_id=path.stem,
            canonical_id=canonical,canonical_status=status,question_hash=h,
            request_verified=bool(question and users and norm(question) in norm(users[-1])),
            final_chars=len(final),final_sha256=sha(final),stripped_final_sha256=sha(final.strip()),
            normalized_final_sha256=sha(norm(final)),reasoning_sha256=sha(norm(reasoning)) if reasoning else '',
            reasoning_only=not final.strip() and bool(reasoning.strip()),api_success=success,
            numeric_only=bool(re.fullmatch(r'[+-]?\d+(?:\.\d+)?',final.strip())),
            assistant_history_turns=len(assistants),assistant_history_words=sum(len(t.split()) for t in assistants),
            source_error=error,raw_path=str(path.relative_to(ROOT)),raw_sha256=hashlib.sha256(binary).hexdigest())
    rows=[]
    with ThreadPoolExecutor(max_workers=8) as pool:
        for i,row in enumerate(pool.map(inspect,tasks),1):
            rows.append(row)
            if i%10000==0 or i==len(tasks): print(benchmark,i,len(tasks),flush=True)
    frame=pd.DataFrame(rows)
    frame['duplicate_canonical_key']=frame.canonical_id.ne('') & frame.duplicated(['model_id','persona','subset','canonical_id'],keep=False)
    frame['pair_eligible']=frame.canonical_id.ne('') & ~frame.duplicate_canonical_key & frame.request_verified & frame.api_success
    save(frame,benchmark.lower()+'_source_audit.csv.gz')
    coverage=frame.groupby(['model_id','persona','subset'],as_index=False).agg(total=('raw_path','size'),invalid_json=('source_error',lambda s:s.ne('').sum()),nonempty_final=('final_chars',lambda s:s.gt(0).sum()),numeric_only=('numeric_only','sum'),reasoning_only=('reasoning_only','sum'),pair_eligible=('pair_eligible','sum'),duplicate_rows=('duplicate_canonical_key','sum'),with_assistant_history=('assistant_history_turns',lambda s:s.gt(0).sum()))
    save(coverage,benchmark.lower()+'_coverage.csv')
    pairing=[]
    for (model,subset),g in frame.groupby(['model_id','subset']):
        for left,right,role in contrasts:
            a=g[g.persona.eq(left)];b=g[g.persona.eq(right)]
            old=a.merge(b,on='original_question_id',suffixes=('_a','_b'),validate='one_to_one')
            valid=old.question_hash_a.ne('') & old.question_hash_b.ne('')
            verified=a[a.pair_eligible].merge(b[b.pair_eligible],on='canonical_id',suffixes=('_a','_b'),validate='one_to_one')
            assert verified.question_hash_a.eq(verified.question_hash_b).all()
            pairing.append(dict(model_id=model,subset=subset,left=left,right=right,role=role,old_pairs=len(old),old_mismatches=int((valid & old.question_hash_a.ne(old.question_hash_b)).sum()),old_unknown=int((~valid).sum()),verified_question_pairs=len(verified)))
    save(pd.DataFrame(pairing),benchmark.lower()+'_pairing.csv')
    if benchmark=='GSM8K':
        old_path=ROOT/'result/benchmarks/gsm8k/reports/Regional_Cultural_Persona/linguistic_markers/regional_gsm8k_full_source_aligned_v1_1_20260818/response_records.csv'
        old=pd.read_csv(old_path,usecols=['model_id','persona','question_id','visible_text'],keep_default_na=False)
        old['old_normalized_text_hash']=old.visible_text.map(lambda t:sha(norm(t)))
        old['old_chars']=old.visible_text.str.len()
        old=old.drop(columns='visible_text')
    else:
        old_path=ROOT/'result/benchmarks/ethics/reports/regional_lm_v3_20260916_r5/response_level.csv.gz'
        old=pd.read_csv(old_path,usecols=['model_id','persona','question_id','raw_final_sha256','text_available','text_status'],keep_default_na=False)
    old=old.rename(columns={'question_id':'original_question_id'})
    comparison=old.merge(frame,on=['model_id','persona','original_question_id'],how='outer',indicator=True,validate='one_to_one')
    if benchmark=='GSM8K':
        comparison['old_equals_reasoning']=comparison.old_chars.gt(0) & comparison.old_normalized_text_hash.eq(comparison.reasoning_sha256)
        comparison['old_nonempty_final_empty']=comparison.old_chars.gt(0) & comparison.final_chars.eq(0)
    else:
        comparison['old_final_matches']=comparison.raw_final_sha256.eq(comparison.final_sha256)|comparison.raw_final_sha256.eq(comparison.stripped_final_sha256)
    save(comparison,benchmark.lower()+'_old_source_comparison.csv.gz')
    print(coverage.groupby('model_id').sum(numeric_only=True).to_string(),flush=True)
    print(pd.DataFrame(pairing).groupby('role').sum(numeric_only=True).to_string(),flush=True)
    if benchmark=='GSM8K': print(comparison[['old_equals_reasoning','old_nonempty_final_empty']].sum().to_dict(),flush=True)
    else: print('OLD FINAL MATCH',comparison.old_final_matches.value_counts().to_dict(),flush=True)
print('AUDIT COMPLETE',OUT,flush=True)


In [ ]:
# Replay diagnostic cases; authored judgments are not generated by this code.
from pathlib import Path
from datetime import datetime
import json, hashlib, dataclasses, random, sys
import pandas as pd
ROOT=Path('D:/TugasAkhir')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.benchmarks.ethics.religion_lm_v3_extract import extract_final
from src.benchmarks.ethics.regional_lm_v3_extract import extract_response
from src.core.linguistic_markers.registry import MarkerSpec, PatternSpec
from src.core.linguistic_markers.extractor import extract_markers
from src.benchmarks.ethics.gender_lm_v2.surfaces import build_surfaces
ROOT=Path('D:/TugasAkhir')
REPLAY=ROOT/'result/analyses/linguistic_markers/runs'/('religion_regional_marker_replay_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
REPLAY.mkdir(parents=True,exist_ok=False)
def final_text(path):
    obj=json.loads(path.read_bytes())
    response=obj.get('response') or {}
    choices=(response.get('raw_api_response') or {}).get('choices') or []
    message=choices[0].get('message',{}) if choices else {}
    value=message.get('content') if 'content' in message else response.get('content')
    return value if isinstance(value,str) else ''
def make_marker(row):
    allowed={f.name for f in dataclasses.fields(MarkerSpec)}
    d={k:v for k,v in row.items() if k in allowed}
    d['patterns']=tuple(PatternSpec(**p) for p in json.loads(d['patterns']))
    d['benchmarks']=tuple(d['benchmarks'].split(';'))
    return MarkerSpec(**d)
snapshots=[]
targets=[
('religion','A002','deepseek_deepseek-r1-distill-qwen-32b','christian_explicit','ethics_commonsense_0162'),
('religion','A014','meta-llama_llama-3.1-8b-instruct','atheist_implicit_2','ethics_deontology_0198'),
('religion','A038','mistralai_ministral-8b-2512','jewish_implicit_1','ethics_justice_0072'),
('regional','R011','mistralai_ministral-8b-2512','north_american_implicit_2','ethics_deontology_0314'),
('regional','R012','mistralai_ministral-8b-2512','neutral','ethics_commonsense_0133'),
('regional','R015','openai_gpt-oss-20b','neutral','ethics_deontology_0255'),
('regional','R016','qwen_qwen3-next-80b-a3b-thinking','north_american_explicit','ethics_justice_0106')]
for dimension,aid,model,persona,qid in targets:
    domain='Neutral Persona' if persona=='neutral' else ('Religion Persona' if dimension=='religion' else 'Regional Cultural Persona')
    base=ROOT/'result/benchmarks/ethics/runs/v9_main'/domain/model
    path=base/persona/(qid+'.json')
    if not path.exists():path=base/(qid+'.json')
    text=final_text(path)
    if dimension=='religion':
        metrics,hits=extract_final(text)
    else:
        result=extract_response(text,rule_set='candidate');metrics,hits=result['metrics'],result['hits']
    snapshots.append(dict(dimension=dimension,benchmark='ETHICS',audit_id=aid,selection='targeted_previous_failure',raw_path=str(path.relative_to(ROOT)),raw_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),final_sha256=hashlib.sha256(text.encode()).hexdigest(),final_text=text,metrics=metrics,hits=hits))
rng=random.Random(20261005)
for dimension,domain,run in [
('religion','Religion_Persona','religion_gsm8k_full_source_aligned_v1_3_20260818'),
('regional','regional_cultural_persona','regional_gsm8k_full_source_aligned_v1_1_20260818')]:
    report_domain='Religion_Persona' if dimension=='religion' else 'Regional_Cultural_Persona'
    reg=pd.read_csv(ROOT/'result/benchmarks/gsm8k/reports'/report_domain/'linguistic_markers'/run/'marker_registry.csv',keep_default_na=False)
    markers=tuple(make_marker(r) for r in reg.to_dict('records'))
    for folder in sorted((ROOT/'result/benchmarks/gsm8k'/domain).iterdir()):
        if not folder.is_dir():continue
        paths=sorted(folder.rglob('gsm8k_*.json'));rng.shuffle(paths)
        for path in paths:
            try:text=final_text(path)
            except ValueError:continue
            if text.strip() and not text.strip().isdigit():break
        surface=build_surfaces(text)
        metrics=extract_markers({'benchmark':'gsm8k','response_content':surface.visible_text},markers)
        snapshots.append(dict(dimension=dimension,benchmark='GSM8K',audit_id='G_'+dimension+'_'+folder.name,selection='seeded_shuffle_first_nonempty_nonnumeric_per_model',raw_path=str(path.relative_to(ROOT)),final_sha256=hashlib.sha256(text.encode()).hexdigest(),final_text=text,metrics=metrics))
rng=random.Random(20261005)
for folder in sorted((ROOT/'result/benchmarks/ethics/runs/v9_main/Regional Cultural Persona').iterdir()):
    if not folder.is_dir():continue
    paths=sorted(folder.rglob('ethics_*.json'));rng.shuffle(paths)
    for path in paths:
        try:text=final_text(path)
        except ValueError:continue
        if len(text.strip())>10:break
    result=extract_response(text,rule_set='candidate')
    snapshots.append(dict(dimension='regional',benchmark='ETHICS',audit_id='E_regional_'+folder.name,selection='seeded_shuffle_first_final_longer_than_10_chars_per_model',raw_path=str(path.relative_to(ROOT)),final_sha256=hashlib.sha256(text.encode()).hexdigest(),final_text=text,metrics=result['metrics'],hits=result['hits']))
assert len(snapshots)==25
(REPLAY/'readback_and_replay.json').write_text(json.dumps(snapshots,ensure_ascii=False,indent=2),encoding='utf-8')
pd.DataFrame([{k:v for k,v in r.items() if k not in ['final_text','metrics','hits']} for r in snapshots]).to_csv(REPLAY/'sample_manifest.csv',index=False,encoding='utf-8-sig')
print('REPLAY',REPLAY,'responses',len(snapshots),flush=True)
for r in snapshots:
    print(r['audit_id'],{k:v for k,v in r['metrics'].items() if k.endswith('__count') and v is not None and v>0},flush=True)
